# Record a dataset session

Drive the line with the gamepad while the robot saves camera frames. Labels are added later on the workstation.

1. Run the cells top to bottom.
2. Fill in the session fields (what you are about to drive).
3. Click **ARM**, click **record**, drive, then click **record** again to finish.
4. Run the last cell before closing the notebook, so the camera is free again.

Frames go to the USB stick: `/workspace/usb/images/datasets/<date>_<name>/`, together with a `session.json`.

In [ ]:
# Gamepad layout (same as drone_teleoperation.ipynb)
CONTROLLER_INDEX = 0
STEERING_AXIS = 0   # left stick, horizontal
THROTTLE_AXIS = 5   # right stick, vertical

# Driving feel
DEADZONE = 0.1       # ignore stick values closer than this to the center
MAX_SPEED = 0.25     # top wheel speed while recording, keep it slow and steady
STEERING_GAIN = 0.5  # how strongly the left stick turns the robot

# Recording
RECORD_HZ = 4.0      # frames saved per second (30 Hz would only give near duplicates)

In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display

controller = widgets.Controller(index=CONTROLLER_INDEX)
display(controller)

Press a button on the gamepad so the browser registers it, then continue.

In [ ]:
from jetbot import Robot

robot = Robot()

arm_button = widgets.ToggleButton(value=False, description='DISARMED', button_style='danger')
left_speed = widgets.FloatText(description='left', disabled=True)
right_speed = widgets.FloatText(description='right', disabled=True)


def apply_deadzone(value):
    """Returns 0 inside the deadzone, and rescales values outside it to the full [-1, 1] range"""
    if abs(value) <= DEADZONE:
        return 0.0
    sign = 1.0 if value > 0 else -1.0
    return sign * (abs(value) - DEADZONE) / (1.0 - DEADZONE)


def arcade_mix(throttle, steering):
    """Mixes throttle and steering into (left, right) wheel speeds"""
    left = throttle + steering * STEERING_GAIN
    right = throttle - steering * STEERING_GAIN
    # if a wheel would go above full speed, scale both wheels down together so the turn keeps its shape
    scale = max(1.0, abs(left), abs(right))
    return MAX_SPEED * left / scale, MAX_SPEED * right / scale


def update_motors(change=None):
    if arm_button.value:
        # gamepads report stick up as negative, so flip it to make up = forward
        throttle = -apply_deadzone(controller.axes[THROTTLE_AXIS].value)
        left, right = arcade_mix(throttle, apply_deadzone(controller.axes[STEERING_AXIS].value))
    else:
        left, right = 0.0, 0.0
    robot.set_motors(left, right)
    left_speed.value, right_speed.value = left, right


def handle_arm(change):
    arm_button.description = 'ARMED' if change['new'] else 'DISARMED'
    arm_button.button_style = 'success' if change['new'] else 'danger'
    update_motors()


arm_button.observe(handle_arm, names='value')
controller.axes[STEERING_AXIS].observe(update_motors, names='value')
controller.axes[THROTTLE_AXIS].observe(update_motors, names='value')

display(widgets.HBox([arm_button, left_speed, right_speed]))

In [ ]:
import traitlets
from jetbot import Camera, bgr8_to_jpeg

camera = Camera.instance()
image_widget = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

display(image_widget)

In [ ]:
from recorder import Recorder

recorder = Recorder(period=1.0 / RECORD_HZ)

name = widgets.Text(description='name', value='line')
tape_color = widgets.Text(description='tape', value='')
lighting = widgets.Dropdown(description='lighting', options=['bright', 'normal', 'dim'], value='normal')
section = widgets.Dropdown(description='section', options=['mixed', 'straight', 'gentle', 'sharp', 'off_line'], value='mixed')
obstacle = widgets.Dropdown(description='obstacle', options=['none', 'on_line', 'beside_line'], value='none')
notes = widgets.Text(description='notes', value='')

record_button = widgets.ToggleButton(value=False, description='record', button_style='')
status = widgets.Label(value='not recording')


def save_frame(change):
    if recorder.offer(image_widget.value):
        status.value = '%d frames -> %s' % (recorder.count, recorder.session_dir)


def handle_record(change):
    if change['new']:
        recorder.start(name.value, {
            'tape_color': tape_color.value,
            'lighting': lighting.value,
            'section': section.value,
            'obstacle': obstacle.value,
            'notes': notes.value,
        })
        camera.observe(save_frame, names='value')
        record_button.button_style = 'danger'
    else:
        camera.unobserve(save_frame, names='value')
        status.value = 'saved %d frames in %s' % (recorder.count, recorder.stop())
        record_button.button_style = ''


record_button.observe(handle_record, names='value')

display(widgets.VBox([name, tape_color, lighting, section, obstacle, notes,
                      widgets.HBox([record_button, status])]))

### Finish

Run this when the session is done. It stops the motors, the recording and the camera.

In [ ]:
record_button.value = False
arm_button.value = False
robot.stop()
camera_link.unlink()
camera.stop()